# Battery Optimization with Price effects Version 1.5.1
Modified the author's code to be able to read in AESO prices. Here I use prices for the year 2023.
NOTE: Still haven't added in Wind and Solar yet.

source:
\
https://medium.com/@yeap0022/basic-build-optimization-model-to-schedule-batterys-operation-in-power-grid-systems-51a8c04b3a0e

### Loading in packages

In [146]:
# Let's load in the necessary packages

import pandas as pd
import datetime
import os
import timeit
import matplotlib.pyplot as plt

from ortools.linear_solver import pywraplp
import numpy as np
pd.set_option('display.max_rows',50)

In [147]:
# Loading in price data

file_name = os.fsdecode("marco_data.csv")
workbook = pd.read_csv(file_name)



In [148]:

merit = workbook.filter(items=["date", "he", "size","flexible","price"]) # Gets the columns that we need
merit = merit.query('date=="2024-04-10" and he==18')
merit
#I'm going to use this to test my function


,date,he,size,flexible,price
4339,2024-04-10,18,0.0041,N,0.00
4340,2024-04-10,18,0.0857,N,0.00
4341,2024-04-10,18,0.1546,N,0.00
4342,2024-04-10,18,0.9536,N,0.00
4343,2024-04-10,18,1.1642,N,0.00
...,...,...,...,...,...
4611,2024-04-10,18,16.0000,Y,999.99
4612,2024-04-10,18,42.0000,N,999.99
4613,2024-04-10,18,80.0000,Y,999.99
4614,2024-04-10,18,109.0000,Y,999.99


In [ ]:
#figure out a price from the merit data
#receive bids (size, price, flex)
def pricing(bids, ail):
  #print("AIL is ",ail)
  #sort bids by price
    
  #merit is the cumulative sum of offered power
  bids=bids.assign(merit=bids['size'].cumsum())
  #any offers for which merit is > ail is a surplus
  bids = bids.assign(surplus=bids['merit']-ail)
  #print(bids['surplus'])
  #the last fully used block is the last block in which surplus is less than or equal to zero
  last_full=bids.query('surplus<=0')['surplus'].idxmax()
  #print(last_full)
  #print(bids.head(50))
  #print(bids.iloc[[0]])
  #you don't need this but keep it if you want
  #bids = bids.assign(dispatched="N")
  #bids['dispatched_mw'] = bids['size']
  #bids.loc[last_full+1:,'dispatched_mw'] = 0
  #bids.loc[:last_full,['dispatched']] = "Y"
  #bids.loc[:last_full,['dispatched_mw']] = bids.loc[:last_full,['size']]
  #the amount of energy I need to supply AIL after I use last_full is the negative of the surplus at that block
  e_needed=bids['surplus'][last_full]*-1
  #print("last fully-used block is",last_full, "and energy still needed is ",e_needed)
  #print(bids.loc[last_full-2:last_full+2])
  while(e_needed>0):
    last_full+=1 #skip to the next block
    #print("Now checking block ",last_full,"Size is",bids['size'][last_full])
    #print(bids.loc[last_full])
    dispatch=0
    if(bids['size'][last_full]<=e_needed): #dispatch it if I can use it all (this will only be true for subsequent blocks other than the first one)
        print("using all of block", last_full)
        dispatch=bids['size'][last_full]  #if you adjust e_needed here, the next if could get messed up
        #also don't need this
        #bids.loc[last_full,['dispatched']] = "Y"
        #bids.loc[last_full,['dispatched_mw']] = dispatch
    elif (bids['surplus'][last_full]>e_needed) and (bids['flexible'][last_full]=="Y"): #dispatch the part you need 
        #if block is too large in not flexible we skip over it because it's not in either if statement
        print("using part of flexible block", last_full)
        dispatch = e_needed #set e_needed to zero as you've now got all the power you need
        #and this is not needed either
        #bids.loc[last_full,['dispatched']] = "Y"
        #bids.loc[last_full,['dispatched_mw']] = dispatch
    e_needed-=dispatch
  price=bids['price'][last_full] #price is the price of the last block used
  #print(bids.loc[last_full-2:last_full+2])
  #print(e_needed,last_full,price)
  return(price) 

bids_testing= merit.filter(items=["size","flexible","price"])
pricing(bids_testing,765)